# Causal GAIL on HumanoidMaze Medium

In [1]:
import random
import copy
import torch
import pickle
import os
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import *
from causal_rl.algo.imitation.gail.causal_gail import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 2000
seed = 0
lookback = 1
hidden_dims = {'V'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, O hidden
train_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, O hidden
eval_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = HumanoidMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
Z_sets = find_sequential_pi_backdoor(G, X_small, Y, obs_prefix)

base_step = small_steps - 1
base_Z_set = Z_sets[f'X{base_step}']

for i in range(base_step + 1, num_steps):
    updated_base_Z_set = set()
    for v in base_Z_set:
        updated_base_Z_set.add(f'{v[0]}{int(v[1:]) + i - lookback}')

    Z_sets[f'X{i}'] = updated_base_Z_set

Z_sets['X1']

{'A0', 'A1', 'C0', 'C1', 'E0', 'E1', 'H0', 'H1', 'J0', 'J1', 'P0', 'P1', 'X0'}

## Expert Trajectories

In [7]:
# for eval: corrupted W, O shown
traj_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True)
# load model
MODEL_PATH = '/home/et2842/causal/causalrl/models/humanoidmaze_medium_expert_finetuned.pt'
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

action_bounds = (ckpt['action_bounds_low'], ckpt['action_bounds_high'])

expert_model = ContinuousPolicyNN(
    input_dim=ckpt['input_dim'],
    action_dim=ckpt['num_actions'],
    hidden_dim=256,
    num_blocks=ckpt['num_blocks'],
    dropout=ckpt['dropout'],
    layernorm=ckpt['layernorm'],
    final_tanh=ckpt['final_tanh'],
    action_bounds=action_bounds,
).to(device)

expert_model.load_state_dict(ckpt['state_dict'])
expert_model.eval()

slots = ckpt['slots']
Z_trim = ckpt['Z_trim']
dims = ckpt['dims']
lookback = ckpt['lookback']

expert_policy = shared_policy_fn_long_horizon(expert_model, slots, Z_trim, continuous=True, device=device)
expert_policies = make_shared_policy_dict(expert_policy)
num_eval_eps = 500

records = collect_imitator_trajectories(
    env=traj_env,
    policies=expert_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    show_progress=True
)

len(records)

Starting episode 1/500...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/500...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/500...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/500...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/500...


...

  Episode 497 ended at step 2000 (terminated: False, truncated: True).
Starting episode 498/500...


  Episode 498 ended at step 2000 (terminated: False, truncated: True).
Starting episode 499/500...


  Episode 499 ended at step 2000 (terminated: False, truncated: True).
Starting episode 500/500...


  Episode 500 ended at step 1352 (terminated: True, truncated: False).
Finished collecting imitator trajectories.


978875

In [8]:
dims = {
    'P': 2,
    'A': 21,
    'H': 1,
    'E': 12,
    # 'V': 3,
    'C': 3,
    'J': 27,
    'W': 2,
    'X': 21
}

In [9]:
sample_obs = records[0]['obs']

# Trim Z-sets to the lookback window (this matches what you do for BC)
causal_Z_trim = trim_Z_sets(Z_sets, lookback=lookback)

# Build windowed encoders that depend on relative lags (not absolute time)
causal_encode, causal_z_dim, causal_slots = build_windowed_z_encoder(
    causal_Z_trim,
    dims=dims,
    lookback=lookback,
)

causal_z_dim

153

In [10]:
# precompute expert batches once (so one_training_round doesn't redo this every time)
Z_e_causal, A_e_causal, X_e_causal = make_expert_batch(records, causal_encode)
X_e_causal = X_e_causal.to(device)

## Hyperparameters

In [11]:
# PPO
gail_gamma          = 0.99
gae_lambda          = 0.95
ppo_clip            = 0.2
ppo_epochs          = 4
ppo_minibatch_size  = 1024
entropy_coeff       = 1e-2
value_coeff         = 0.5
max_grad_norm       = 0.5
normalize_adv       = True

# discriminator
d_loss_type         = 'bce'
gp_lambda           = 5.0
d_updates           = 2
d_minibatch_size    = 1024
use_gp              = True
instance_noise_std  = 0.0
label_smoothing     = 0.0

# rollout
max_steps_per_episode   = num_steps
episodes_per_round      = 20
num_rounds_causal_gail  = 500

# network
hidden_size_actor   = 256
hidden_size_critic  = 256
hidden_size_disc    = 256
actor_lr            = 1e-4
critic_lr           = 3e-4
disc_lr             = 3e-4
num_blocks_actor    = 3
dropout_actor       = 0.05
layernorm_actor     = True

## Network Initialization

In [12]:
action_dim = train_env.env.action_space.shape[0]
action_low = float(train_env.env.action_space.low.min())
action_high = float(train_env.env.action_space.high.max())

causal_actor = ContinuousActor(
    num_inputs=causal_z_dim,
    num_outputs=action_dim,
    hidden_size=hidden_size_actor,
    std=0.0,
    action_low=action_low,
    action_high=action_high,
    num_blocks=num_blocks_actor,
    dropout=dropout_actor,
    layernorm=layernorm_actor,
).to(device)

causal_critic = Critic(
    num_inputs=causal_z_dim,
    hidden_size=hidden_size_critic,
).to(device)

causal_disc = Discriminator(
    num_inputs=causal_z_dim + action_dim,
    hidden_size=hidden_size_disc,
    dropout=0.2,
).to(device)

actor_optim_causal = torch.optim.Adam(causal_actor.parameters(), lr=actor_lr)
critic_optim_causal = torch.optim.Adam(causal_critic.parameters(), lr=critic_lr)
disc_optim_causal = torch.optim.Adam(causal_disc.parameters(), lr=disc_lr)

## Training

In [13]:
best_return = -float('inf')
best_actor_sd = None
return_window = []
WINDOW = 20

disc_scheduler = torch.optim.lr_scheduler.StepLR(disc_optim_causal, step_size=100, gamma=0.5)

logs_causal_gail = []

for it in range(1, num_rounds_causal_gail + 1):
    stats = one_training_round(
        env=train_env,
        actor=causal_actor,
        critic=causal_critic,
        discriminator=causal_disc,
        actor_optim=actor_optim_causal,
        critic_optim=critic_optim_causal,
        discriminator_optim=disc_optim_causal,
        encode=causal_encode,
        X_e=X_e_causal,
        expert_records=None,
        gamma=gail_gamma,
        gae_lambda=gae_lambda,
        ppo_clip=ppo_clip,
        epochs=ppo_epochs,
        minibatch_size=ppo_minibatch_size,
        entropy_coeff=entropy_coeff,
        value_coeff=value_coeff,
        max_grad_norm=max_grad_norm,
        normalize_adv=normalize_adv,
        loss_type=d_loss_type,
        gp_lambda=gp_lambda,
        d_updates=d_updates,
        d_minibatch_size=d_minibatch_size,
        use_gp=use_gp,
        instance_noise_std=instance_noise_std,
        label_smoothing=label_smoothing,
        max_steps=max_steps_per_episode,
        num_episodes=episodes_per_round,
        seed=seed + it
    )
    logs_causal_gail.append(stats)
    disc_scheduler.step()

    # rolling average return tracking
    return_window.append(stats['avg_env_return'])
    if len(return_window) > WINDOW:
        return_window.pop(0)
    avg_ret = sum(return_window) / len(return_window)

    if avg_ret > best_return:
        best_return = avg_ret
        best_actor_sd = copy.deepcopy(causal_actor.state_dict())

    if it % 10 == 0:
        print(
            f"[Causal GAIL iter {it}] "
            f"return={stats['avg_env_return']:.2f}, "
            f"D_loss={stats['D_loss']:.3f}, "
            f"actor_loss={stats['ppo_actor_loss']:.3f}, "
            f"best_avg={best_return:.2f}"
        )

# restore best checkpoint
causal_actor.load_state_dict(best_actor_sd)
print(f"Restored best checkpoint (avg return={best_return:.2f})")

[Causal GAIL iter 10] return=-745.10, D_loss=0.093, actor_loss=-0.317, best_avg=-704.90


[Causal GAIL iter 20] return=-808.38, D_loss=0.082, actor_loss=-0.312, best_avg=-704.90


[Causal GAIL iter 30] return=-775.35, D_loss=0.090, actor_loss=-0.301, best_avg=-704.90


[Causal GAIL iter 40] return=-664.78, D_loss=0.097, actor_loss=-0.288, best_avg=-704.90


[Causal GAIL iter 50] return=-710.91, D_loss=0.109, actor_loss=-0.281, best_avg=-703.60


...

[Causal GAIL iter 460] return=-740.99, D_loss=0.239, actor_loss=-0.203, best_avg=-659.67


[Causal GAIL iter 470] return=-729.28, D_loss=0.221, actor_loss=-0.234, best_avg=-659.67


[Causal GAIL iter 480] return=-732.92, D_loss=0.212, actor_loss=-0.212, best_avg=-659.67


[Causal GAIL iter 490] return=-716.26, D_loss=0.233, actor_loss=-0.217, best_avg=-659.67


[Causal GAIL iter 500] return=-708.63, D_loss=0.260, actor_loss=-0.208, best_avg=-659.67
Restored best checkpoint (avg return=-659.67)


## Evaluation

In [14]:
causal_gail_policy = make_gail_policy(causal_actor, causal_encode, device=device, deterministic=True)
causal_gail_policies = make_shared_policy_dict(causal_gail_policy)

In [15]:
num_eval_eps = 10
causal_gail_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=causal_gail_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    seed=seed + 90210,
    show_progress=True
)

len(causal_gail_returns)

Starting episode 1/10...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/10...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/10...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/10...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/10...


...

  Episode 7 ended at step 2000 (terminated: False, truncated: True).
Starting episode 8/10...


  Episode 8 ended at step 2000 (terminated: False, truncated: True).
Starting episode 9/10...


  Episode 9 ended at step 2000 (terminated: False, truncated: True).
Starting episode 10/10...


  Episode 10 ended at step 2000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


20000

In [16]:
causal_gail_episode_rewards = defaultdict(float)
for rec in causal_gail_returns:
    ep = rec['episode']
    causal_gail_episode_rewards[ep] += float(rec['reward'])

causal_gail_rewards = [causal_gail_episode_rewards[e] for e in range(num_eval_eps)]
sum(causal_gail_rewards) / num_eval_eps

-837.5364189377764

## Save Model

In [17]:
# save model
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, 'cgail_hummed.pt')

causal_gail_ckpt = {
    "state_dict": causal_actor.state_dict(),
    "z_dim": causal_z_dim,
    "action_dim": action_dim,
    "hidden_size_actor": hidden_size_actor,
    "num_blocks_actor": num_blocks_actor,
    "dropout_actor": dropout_actor,
    "layernorm_actor": layernorm_actor,
    "final_tanh": True,
    "action_bounds_low": eval_env.env.action_space.low,
    "action_bounds_high": eval_env.env.action_space.high,
    "Z_sets": causal_Z_trim,
    "dims": dims,
    "lookback": lookback,
}

torch.save(causal_gail_ckpt, MODEL_PATH)
print("Saved Causal GAIL actor to:", MODEL_PATH)

Saved Causal GAIL actor to: /home/et2842/causal/causalrl/models/cgail_hummed.pt
